# azure_monitor_opentelemetry_exporter smoke test

Verifies the Azure Monitor OpenTelemetry exporter imports + its primary API surface works.

In [ ]:
# Example 1: import the three primary exporter classes (traces, metrics, logs).
import os

# Keep every example offline: switch off the exporter's own usage reporting (statsbeat and
# SDK stats), which would otherwise send metrics to Azure Monitor in the background.
os.environ["APPLICATIONINSIGHTS_STATSBEAT_DISABLED_ALL"] = "true"
os.environ["APPLICATIONINSIGHTS_SDKSTATS_DISABLED"] = "true"

from azure.monitor.opentelemetry.exporter import (
    AzureMonitorTraceExporter,
    AzureMonitorMetricExporter,
    AzureMonitorLogExporter,
)

print(f"AzureMonitorTraceExporter: {AzureMonitorTraceExporter}")
print(f"AzureMonitorMetricExporter: {AzureMonitorMetricExporter}")
print(f"AzureMonitorLogExporter: {AzureMonitorLogExporter}")

assert callable(AzureMonitorTraceExporter)
assert callable(AzureMonitorMetricExporter)
assert callable(AzureMonitorLogExporter)

In [ ]:
# Example 2: construct a trace exporter with a fake connection string.
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter

fake_conn_str = (
    "InstrumentationKey=00000000-0000-0000-0000-000000000000;"
    "IngestionEndpoint=https://westus2-0.in.applicationinsights.azure.com/"
)

exporter = AzureMonitorTraceExporter(connection_string=fake_conn_str)
print(f"Trace exporter: {exporter}")
print(f"Has export method: {hasattr(exporter, 'export')}")
print(f"Has shutdown method: {hasattr(exporter, 'shutdown')}")

assert hasattr(exporter, 'export')
assert hasattr(exporter, 'shutdown')
exporter.shutdown()

In [ ]:
# Example 3: construct a metric exporter and check the OpenTelemetry SDK interface.
from azure.monitor.opentelemetry.exporter import AzureMonitorMetricExporter
from opentelemetry.sdk.metrics.export import MetricExporter

fake_conn_str = (
    "InstrumentationKey=00000000-0000-0000-0000-000000000000;"
    "IngestionEndpoint=https://westus2-0.in.applicationinsights.azure.com/"
)

metric_exporter = AzureMonitorMetricExporter(connection_string=fake_conn_str)
print(f"Metric exporter: {metric_exporter}")
print(f"Is MetricExporter subclass: {isinstance(metric_exporter, MetricExporter)}")

assert isinstance(metric_exporter, MetricExporter)
assert hasattr(metric_exporter, 'export')
metric_exporter.shutdown()

In [ ]:
# Example 4: construct a log exporter and verify its LogRecordExporter interface.
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter
# AzureMonitorLogExporter implements the OpenTelemetry SDK's LogRecordExporter interface.
from opentelemetry.sdk._logs.export import LogRecordExporter

fake_conn_str = (
    "InstrumentationKey=00000000-0000-0000-0000-000000000000;"
    "IngestionEndpoint=https://westus2-0.in.applicationinsights.azure.com/"
)

log_exporter = AzureMonitorLogExporter(connection_string=fake_conn_str)
print(f"Log exporter: {log_exporter}")
print(f"Is LogRecordExporter subclass: {isinstance(log_exporter, LogRecordExporter)}")

assert isinstance(log_exporter, LogRecordExporter)
assert hasattr(log_exporter, 'export')
log_exporter.shutdown()

In [ ]:
# Example 5: export a span through a TracerProvider and inspect the uploaded envelope.
# The exporter's HTTP transport is a requests session whose CannedAdapter accepts each upload,
# so the span is serialized and posted without leaving the process.
import http
import io
import json

import requests
from azure.core.pipeline.transport import RequestsTransport
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter


class CannedAdapter(requests.adapters.BaseAdapter):
    def __init__(self, handler):
        super().__init__()
        self.handler = handler  # request -> (status code, JSON payload)
        self.requests = []

    def send(self, request, **kwargs):
        self.requests.append(request)
        status_code, payload = self.handler(request)
        body = json.dumps(payload).encode()
        response = requests.Response()
        response.status_code = status_code
        response.reason = http.HTTPStatus(status_code).phrase
        response.headers = requests.structures.CaseInsensitiveDict(
            {"Content-Type": "application/json", "Content-Length": str(len(body))})
        response.raw = io.BytesIO(body)
        response.url = request.url
        response.request = request
        return response

    def close(self):
        pass


def ingestion(request):
    count = len(json.loads(request.body))
    return 200, {"itemsReceived": count, "itemsAccepted": count, "errors": []}


fake_conn_str = (
    "InstrumentationKey=00000000-0000-0000-0000-000000000000;"
    "IngestionEndpoint=https://westus2-0.in.applicationinsights.azure.com/"
)

adapter = CannedAdapter(ingestion)
session = requests.Session()
session.mount("https://", adapter)
provider = TracerProvider()
exporter = AzureMonitorTraceExporter(
    connection_string=fake_conn_str,
    disable_offline_storage=True,
    tracer_provider=provider,
    transport=RequestsTransport(session=session),
)
assert isinstance(exporter, SpanExporter)
provider.add_span_processor(SimpleSpanProcessor(exporter))
tracer = provider.get_tracer("smoke-test")

with tracer.start_as_current_span("test-span") as span:
    span.set_attribute("smoke.test", True)
    assert span.get_span_context().is_valid

(upload,) = adapter.requests
assert upload.method == "POST" and upload.url.endswith("/v2.1/track"), upload.url
envelopes = json.loads(upload.body)
spans = [e for e in envelopes if e["data"]["baseType"] == "RemoteDependencyData"]
assert len(spans) == 1, [e["data"]["baseType"] for e in envelopes]
(envelope,) = spans
assert envelope["iKey"] == "00000000-0000-0000-0000-000000000000"
assert envelope["data"]["baseData"]["name"] == "test-span"
assert envelope["data"]["baseData"]["properties"] == {"smoke.test": "True"}
print(f"uploaded {len(envelopes)} envelopes; span envelope: {envelope['name']}")
provider.shutdown()